# SegNet（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
SegNetは，画像分類ネットワークの一部（VGG16）をエンコーダとして利用しつつ，デコーダ側では転置畳み込みではなく，エンコーダの各`MaxPool2d`で記録した位置情報（インデックス）を使って`MaxUnpool2d`によりアップサンプリングを行うセマンティックセグメンテーションモデルです．FCNのようにスコアマップを補間・加算するのではなく，プーリング時の位置情報をそのまま復元に使う点が特徴です．本ノートブックでは，このエンコーダ・デコーダ構造とプーリングインデックスの受け渡しの仕組みを実装しながら理解します．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCSegmentation
from torchvision.models import vgg16, VGG16_Weights
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．20クラスの物体に対して画素単位のクラスラベルが付与されたデータセットで，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます（セグメンテーション用アノテーションが存在する画像のみのため，画像分類・物体検出のノートブックと比べて画像枚数が少なくなっています）．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．マスク画像は，各画素の値がクラスID（0が背景，1〜20が物体クラス，255が物体の境界など評価から除外する「無視領域」）になっている画像として与えられます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## SegNet
SegNetは，エンコーダに事前学習済みVGG16と同じ構造を用い，デコーダはエンコーダと対称な構造を持つセマンティックセグメンテーションモデルです．FCNは各段のスコアマップをアップサンプリングして加算するスキップ接続でしたが，SegNetはスコアの加算を行わず，デコーダ側の`MaxUnpool2d`にエンコーダの`MaxPool2d`が記録した「どの位置が最大値だったか」というインデックスを渡すことで，位置情報を保ったままアップサンプリングします．転置畳み込み（学習可能なパラメータで補間する）と異なり，`MaxUnpool2d`は該当インデックスの位置に値を書き戻し，それ以外を0で埋めるだけの決定的な演算であるため，デコーダのパラメータ数を抑えつつシャープな境界を再現しやすいという特徴があります．

### エンコーダ（VGG16）
`torchvision`の事前学習済みVGG16の`features`をそのまま使うと，内部の`MaxPool2d`が`return_indices=True`になっておらずインデックスを取得できません．そこで，VGG16の畳み込み部分と同じ構成（畳み込み13層＋5回のプーリング）を自前で構築し，`state_dict`を使って畳み込み層の重みだけをコピーします．原論文のSegNetは各畳み込みの後にBatch Normalizationを加えた構成になっているため（`vgg16`にはBatch Normalizationはない），本ノートブックでもConv-BN-ReLUの並びにしていますが，コピーするのは畳み込み層の重みのみで，Batch Normalizationはランダム初期化のままです．各`MaxPool2d`は`return_indices=True`に設定し，プーリング直前の特徴マップのサイズとインデックスを各段で保存しておきます．

In [ ]:
class SegNetEncoder(nn.Module):
    """VGG16のconv部分と同じ構成を持つエンコーダ（return_indices=Trueのプーリングでインデックスを記録）"""
    def __init__(self):
        super().__init__()
        # VGG16の各畳み込みブロック（チャンネル数・畳み込み回数）と同じ構成
        cfg = [(2, 64), (2, 128), (3, 256), (3, 512), (3, 512)]
        in_ch = 3
        self.blocks = nn.ModuleList()
        for n_conv, out_ch in cfg:
            layers = []
            for _ in range(n_conv):
                layers += [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
                in_ch = out_ch
            self.blocks.append(nn.Sequential(*layers))
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)

    def forward(self, x):
        indices_list = []
        sizes_list = []
        for block in self.blocks:
            x = block(x)
            sizes_list.append(x.size())
            x, indices = self.pool(x)
            indices_list.append(indices)
        return x, indices_list, sizes_list


def load_pretrained_vgg16_weights(encoder):
    """事前学習済みVGG16のconv層の重みを，同じ形状のSegNetEncoderへコピーする"""
    vgg_features = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
    vgg_convs = [m for m in vgg_features if isinstance(m, nn.Conv2d)]

    encoder_convs = [m for block in encoder.blocks for m in block if isinstance(m, nn.Conv2d)]
    assert len(vgg_convs) == len(encoder_convs), 'conv層の数が一致しません'
    for vc, ec in zip(vgg_convs, encoder_convs):
        assert vc.weight.shape == ec.weight.shape, f'形状不一致: {vc.weight.shape} vs {ec.weight.shape}'
        ec.weight.data.copy_(vc.weight.data)
        ec.bias.data.copy_(vc.bias.data)
    return encoder

### デコーダ
デコーダはエンコーダと対称な5段の構成で，各段の先頭で`nn.MaxUnpool2d`を使い，対応するエンコーダの段で保存したインデックスと直前の特徴マップサイズ（`output_size`）を使ってアンプーリングします．アンプーリング後は，チャンネル数を合わせる畳み込みブロックを適用します．最終段の出力に対して1×1畳み込みを適用し，クラス数チャンネルのスコアマップに変換します．

In [ ]:
class SegNetDecoder(nn.Module):
    """エンコーダと対称な構造を持つデコーダ．MaxUnpool2dでインデックスを使ったアップサンプリングを行う"""
    def __init__(self, n_class=n_class):
        super().__init__()
        # (畳み込み回数, 出力チャンネル数) をエンコーダと逆順に並べたもの
        cfg = [(3, 512), (3, 256), (3, 128), (2, 64), (2, 64)]
        in_ch = 512
        self.blocks = nn.ModuleList()
        for n_conv, out_ch in cfg:
            layers = []
            for i in range(n_conv):
                o = out_ch
                layers += [nn.Conv2d(in_ch, o, kernel_size=3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]
                in_ch = o
            self.blocks.append(nn.Sequential(*layers))
        self.unpool = nn.MaxUnpool2d(kernel_size=2, stride=2)
        self.classifier = nn.Conv2d(64, n_class, kernel_size=1)

    def forward(self, x, indices_list, sizes_list):
        # エンコーダとは逆順（深い段から浅い段へ）にインデックスを使ってアンプーリングする
        for block, indices, size in zip(self.blocks, reversed(indices_list), reversed(sizes_list)):
            x = self.unpool(x, indices, output_size=size)
            x = block(x)
        return self.classifier(x)


class SegNet(nn.Module):
    def __init__(self, n_class=n_class):
        super().__init__()
        self.encoder = load_pretrained_vgg16_weights(SegNetEncoder())
        self.decoder = SegNetDecoder(n_class=n_class)

    def forward(self, x):
        x, indices_list, sizes_list = self.encoder(x)
        return self.decoder(x, indices_list, sizes_list)

## 損失関数
セグメンテーションは，画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用できます．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
SegNetを構築し，最適化手法としてAdamを設定します．エンコーダには事前学習済みの重みを使用しているため，スクラッチ学習の分類ノートブック群（`classification/`）よりも小さい学習率を用います．

In [ ]:
model = SegNet(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてSegNetを学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoU（Intersection over Union）の平均であるmIoU（mean IoU）を求めます．mIoUの算出には，`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します．

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのSegNetとの違い

| 項目 | オリジナル論文 (Badrinarayanan et al., 2015/2017) | 本ノートブック |
| :-- | :-- | :-- |
| エンコーダの重み | 事前学習済みVGG16のconv部分の重みで初期化 | 同様に事前学習済みVGG16のconv部分の重みで初期化（FC層は不使用） |
| エンコーダのBatch Normalization | ランダム初期化して学習 | 同様にランダム初期化して学習（`vgg16`自体にはBNがないため，コピー対象は畳み込み層の重みのみ） |
| デコーダの重み | ランダム初期化 | ランダム初期化 |
| 学習データ量 | PASCAL VOCの拡張データ（SBD）を含む大規模データ | VOC2007のtrainvalのみ（422枚） |
| 後処理 | なし | なし（CRFなどの後処理は行わない） |

## 課題

1. `epoch_num`や学習率を変更し，mIoUがどのように変化するか確認してください．
2. `MaxUnpool2d`によるアップサンプリングを，FCNのような転置畳み込みや双線形補間に置き換えた場合，パラメータ数や境界の再現性がどのように変化するか比較してください．
3. デコーダの各段のバッチ正規化（`BatchNorm2d`）を除去した場合，学習の収束や精度にどのような影響があるか確認してください．